# Bird Identification using CV - Part 5.5: Classification preparation

**ITAI 1378  |  Midterm  |  2026**

**Group 8**

**Author:** Stuart Fairchild | Kalen Foster | Ranveer Chand


---

##  Prepare bird classification dataset

Using YOLO11Large, prepare the classification dataset for training the classification model - create bounding box and label with bbox verticies

In [ ]:
%pip install -q ultralytics

from ultralytics import YOLO, SAM
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

print("Setup complete. Ready to detect and segment.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup complete. Ready to detect and segment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Run subfolders individually for adding more species

In [ ]:
import os
base_path = '/content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset'
bird = "Red-bellied Woodpecker"
images_path = f"{base_path}/raw/{bird}"

for filename in os.listdir(images_path):
  if filename.endswith((".png", ".jpg")):
    file_path = os.path.join(images_path, filename)
    print(f"Current file: {filename}")
    # Uncomment below if you want to load all images for viewing
    # img = Image.open(file_path)
    # plt.figure(figsize=(8, 10))
    # plt.imshow(img)
    # plt.axis("off")
    # plt.show()

Current file: 3754.Red-bellied_Woodpecker_0_80_72_690_683_-_f_0037.jpg
Current file: 3754.Red-bellied_Woodpecker_0_124_74_716_666_-_g_0329.jpg
Current file: 3754.Red-bellied_Woodpecker_0_73_258_824_1009_-_f_1134.jpg
Current file: 3754.Red-bellied_Woodpecker_0_214_-35_1024_774_-_f_1559.jpg
Current file: 3754.Red-bellied_Woodpecker_0_-56_228_559_844_-_f_0140.jpg
Current file: 3754.Red-bellied_Woodpecker_0_-10_99_389_500_-_g_0067.jpg
Current file: 3754.Red-bellied_Woodpecker_0_317_149_776_608_-_f_1431.jpg
Current file: 3754.Red-bellied_Woodpecker_0_120_157_986_1024_-_f_1465.jpg
Current file: 3754.Red-bellied_Woodpecker_0_197_93_859_755_-_f_0832.jpg
Current file: 3754.Red-bellied_Woodpecker_0_33_156_644_768_-_f_0277.jpg
Current file: 3754.Red-bellied_Woodpecker_0_-11_58_124_194_-_g_0126.jpg
Current file: 3754.Red-bellied_Woodpecker_0_383_310_884_811_-_g_0400.jpg
Current file: 3754.Red-bellied_Woodpecker_0_319_35_944_660_-_f_0638.jpg
Current file: 3754.Red-bellied_Woodpecker_0_60_138_643_72

## 7. Running detection with YOLO11 Large


In [ ]:
detector = YOLO("yolo11l.pt")
print(f"{bird} loaded.")

detected_path = f"{base_path}/preprocessed/{bird}"
# Ensure the output directory exists
os.makedirs(detected_path, exist_ok=True)

for filename in os.listdir(images_path):
  if filename.endswith((".png", ".jpg")):
    file_path = os.path.join(images_path, filename)
    print(f"Current file: {filename}")
    results = detector.predict(
        source=file_path,
        conf=0.12,           # Set confidence threshold lower
        classes=[14]        # 14 is the COCO index for 'bird'
    )

    # Check if exactly one bird is detected
    boxes = results[0].boxes
    num_birds_detected = len(boxes)

    if num_birds_detected == 1:
        # Get the original image dimensions
        img_width, img_height = results[0].orig_shape[1], results[0].orig_shape[0]

        # Extract bounding box in xywh format (normalized)
        # YOLO boxes are already normalized (center_x, center_y, width, height)
        # box.xywhn[0] returns [center_x_norm, center_y_norm, width_norm, height_norm]
        # We also need the class label, which is 14 for 'bird'
        box_data = boxes.xywhn[0].tolist()
        class_id = boxes.cls[0].item() # Get class ID

        # Format for YOLO .txt file: class_id center_x center_y width height
        bbox_line = f"0 {box_data[0]:.6f} {box_data[1]:.6f} {box_data[2]:.6f} {box_data[3]:.6f}"

        # Define output .txt file path
        txt_filename = os.path.splitext(filename)[0] + ".txt"
        output_txt_file = os.path.join(detected_path, txt_filename)

        # Write bounding box data to file
        with open(output_txt_file, "w") as f:
            f.write(bbox_line)
        print(f"Bounding box coordinates saved to: {output_txt_file}")

        # Plot results without labels and convert from BGR to RGB
        annotated = results[0].plot(labels=False, conf=False)
        annotated_rgb = annotated[..., ::-1]

        output_img_file = os.path.join(detected_path, filename)
        print(f"Output detected image file: {output_img_file}")

        # Save the annotated image directly using Pillow to retain original size
        Image.fromarray(annotated_rgb).save(output_img_file)
    else:
        print(f"Skipping {filename}: {num_birds_detected} birds detected (expected 1).")

Streaming output truncated to the last 5000 lines.
Speed: 3.1ms preprocess, 23.6ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
Skipping 3754.Red-bellied_Woodpecker_0_-20_88_672_781_-_f_0564.jpg: 2 birds detected (expected 1).
Current file: 3754.Red-bellied_Woodpecker_0_262_89_739_567_-_f_1409.jpg

image 1/1 /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/raw/Red-bellied Woodpecker/3754.Red-bellied_Woodpecker_0_262_89_739_567_-_f_1409.jpg: 640x640 1 bird, 24.8ms
Speed: 2.1ms preprocess, 24.8ms inference, 2.2ms postprocess per image at shape (1, 3, 640, 640)
Bounding box coordinates saved to: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/Red-bellied Woodpecker/3754.Red-bellied_Woodpecker_0_262_89_739_567_-_f_1409.txt
Output detected image file: /content/drive/MyDrive/Colab Notebooks/ITAI1378/ITAI1378-midterm/dataset/preprocessed/Red-bellied Woodpecker/3754.Red-bellied_Woodpecker_0_262_89_739_567_-_f_140